# Building Age Classification from Satellite Imagery

# Notebook 4 — Modelling & Evaluation

---

**Approach: Metric Learning**

In this notebook we will use a **metric learning** approach to the classification problem. This involves learning a feature representation where
buildings of the same age class cluster together, regardless of which city they come from. 

In practice this means:
1. **Train on Madrid** (the labelled city) using cross-validation to get reliable performance
   estimates with error bars.
2. **Zero-shot transfer** — apply the Madrid model directly to Amsterdam, with no Amsterdam
   labels, to measure the baseline domain gap.
3. **Few-shot prototype transfer** — given a small number of labelled Amsterdam examples,
   compute per-class prototype vectors (class means in feature space) and classify by
   nearest prototype. This is the inference step of prototype networks, a standard
   metric learning technique.

The few-shot experiment shows how much label budget is needed to close the Madrid–Amsterdam gap.

For more information on the metric learning approach, please see the supportind document "METRIC_LEARNING_APPROACH.md"

**NOTE**:  Metric learning is _one possible approach_.  Feel free to try another approach!

## 4.1 Setup & Imports

In [5]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from xgboost import XGBClassifier

CLASS_COLORS = {1: '#4393c3', 2: '#2ca25f', 3: '#fd8d3c', 4: '#de2d26'}
CLASS_LABELS_MADRID    = {1: 'Pre-1960',  2: '1960-1984', 3: '1984-2004', 4: '2004-2024'}
CLASS_LABELS_AMSTERDAM = {1: 'Pre-1945',  2: '1945-1984', 3: '1984-2004', 4: '2004-2024'}

plt.rcParams['figure.dpi'] = 100
print("Setup complete.")


Setup complete.


## 4.2 Load Preprocessed Data

In [6]:
with open('data/preprocessed/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_madrid    = data['X_madrid']
y_madrid    = data['y_madrid']
X_amsterdam = data['X_amsterdam']
y_amsterdam = data['y_amsterdam']
feature_names = data['feature_names']
pixel_ids_madrid    = data['pixel_ids_madrid']     # (N, 2)  px_key, py_key
pixel_ids_amsterdam = data['pixel_ids_amsterdam']  # (N, 2)  px_key, py_key

print(f'Madrid:    {X_madrid.shape}  ({len(np.unique(y_madrid))} classes)')
print(f'Amsterdam: {X_amsterdam.shape}  ({len(np.unique(y_amsterdam))} classes)')
print(f'Features:  {len(feature_names)}')
print()
for name, y, labels in [('Madrid', y_madrid, CLASS_LABELS_MADRID),
                         ('Amsterdam', y_amsterdam, CLASS_LABELS_AMSTERDAM)]:
    print(f'{name} class distribution:')
    for cls in sorted(np.unique(y)):
        n = (y == cls).sum()
        print(f'  Class {cls} ({labels[cls]}): {n:,}  ({n/len(y)*100:.1f}%)')
    print()
    
smote = SMOTE(random_state=42)
X_madrid, y_madrid = smote.fit_resample(X_madrid, y_madrid)
y_madrid = y_madrid - 1
y_amsterdam = y_amsterdam - 1

Madrid:    (76263, 60)  (4 classes)
Amsterdam: (25992, 60)  (4 classes)
Features:  60

Madrid class distribution:
  Class 1 (Pre-1960): 13,794  (18.1%)
  Class 2 (1960-1984): 27,152  (35.6%)
  Class 3 (1984-2004): 18,672  (24.5%)
  Class 4 (2004-2024): 16,645  (21.8%)

Amsterdam class distribution:
  Class 1 (Pre-1945): 7,568  (29.1%)
  Class 2 (1945-1984): 8,802  (33.9%)
  Class 3 (1984-2004): 6,745  (26.0%)
  Class 4 (2004-2024): 2,877  (11.1%)



In [ ]:
S

### XGBOOST BASELINE MODEL

In [ ]:
# create a baseline xgboost model 
xgb_baseline = XGBClassifier(
    objective: "multi:softprob",
)